In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from rtmlib import Wholebody

import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
PATH         = r"data\Videos"
OUTPUT_DIR   = r"landmarks"
BODY_POINTS  = 17
HAND_POINTS  = 21
BATCH_SIZE   = 128
CONF_THR     = 0.3
MIN_VISIBLE  = 10

LEFT_HAND_SLICE  = slice(91, 112)
RIGHT_HAND_SLICE = slice(112, 133)
MAX_GAP = 1

In [3]:
def normalize_hand(points: np.ndarray) -> np.ndarray:

    out = points.copy()
    xy  = out[:, :2]              
    xy -= xy[0]                   
    scale = np.max(np.linalg.norm(xy, axis=1))  
    if scale > 0:
        xy /= scale               
    return out

In [4]:

def _pick_person(keypoints: np.ndarray, scores: np.ndarray):
    if keypoints.shape[0] == 0:
        return None, None
    idx = int(np.argmax(scores[:, :17].mean(axis=1)))
    return keypoints[idx], scores[idx]


def _body_array(kps: np.ndarray, sc: np.ndarray,
                frame_w: int, frame_h: int) -> np.ndarray:
    body = np.zeros((BODY_POINTS, 3), dtype=np.float32)
    body[:, 0] = kps[:BODY_POINTS, 0] / frame_w   # x in [0, 1]
    body[:, 1] = kps[:BODY_POINTS, 1] / frame_h   # y in [0, 1]
    body[:, 2] = sc[:BODY_POINTS]

    l_conf = sc[5]
    r_conf = sc[6]

    if l_conf > CONF_THR and r_conf > CONF_THR:
        shoulder_l = body[5, :2]
        shoulder_r = body[6, :2]
        midpoint       = (shoulder_l + shoulder_r) * 0.5
        shoulder_width = np.linalg.norm(shoulder_l - shoulder_r)

        if shoulder_width > 1e-4:
            body[:, :2] = (body[:, :2] - midpoint) / shoulder_width

    return body



def _hand_array(kps: np.ndarray, sc: np.ndarray,
                hand_slice: slice,
                frame_w: int, frame_h: int) -> np.ndarray:

    hand_kps = kps[hand_slice]   # (21, 2)
    hand_sc  = sc[hand_slice]    # (21,)

    if (hand_sc > CONF_THR).sum() < MIN_VISIBLE:
        return np.zeros((HAND_POINTS, 3), dtype=np.float32)

    pts = np.zeros((HAND_POINTS, 3), dtype=np.float32)
    pts[:, 0] = hand_kps[:, 0] / frame_w
    pts[:, 1] = hand_kps[:, 1] / frame_h
    pts[:, 2] = hand_sc
    
    pts[hand_sc <= CONF_THR, :2] = pts[0, :2]

    return normalize_hand(pts)

In [5]:
def interpolate_missing_hands(arr: np.ndarray) -> np.ndarray:
    T = len(arr)
    for hand_slice in [slice(17, 38), slice(38, 59)]:
        i = 0
        while i < T:
            if np.all(arr[i, hand_slice] == 0):
                gap_start = i
                j = i + 1
                while j < T and np.all(arr[j, hand_slice] == 0):
                    j += 1
                gap_end = j
                gap_len = gap_end - gap_start
                if gap_len <= MAX_GAP and gap_start > 0 and gap_end < T:
                    before = arr[gap_start - 1, hand_slice]
                    after  = arr[gap_end,        hand_slice]
                    for k in range(gap_len):
                        t = (k + 1) / (gap_len + 1)
                        arr[gap_start + k, hand_slice] = (1 - t) * before + t * after
                i = gap_end
            else:
                i += 1
    return arr

In [6]:
def extract_landmarks_batch(frames: list, wholebody) -> list:
    output = []
    for frame in frames:
        frame_h, frame_w = frame.shape[:2]
        keypoints, scores = wholebody(frame)
        kps, sc = _pick_person(keypoints, scores)

        if kps is None:
            output.append(np.zeros((59, 3), dtype=np.float32))
            continue

        body       = _body_array(kps, sc, frame_w, frame_h) #type: ignore
        left_hand  = _hand_array(kps, sc, LEFT_HAND_SLICE,  frame_w, frame_h) #type: ignore
        right_hand = _hand_array(kps, sc, RIGHT_HAND_SLICE, frame_w, frame_h) #type: ignore

        output.append(np.concatenate([body, left_hand, right_hand], axis=0))  # (59, 3)
    return output

In [7]:
def extract_landmarks_from_video(video_path, wholebody):
    print(f"Processing: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  Could not open: {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    all_frames, batch = [], []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        batch.append(frame)
        frame_idx += 1

        if len(batch) == BATCH_SIZE:
            all_frames.extend(extract_landmarks_batch(batch, wholebody))
            batch = []

        if frame_idx % 100 == 0:
            print(f"  [{os.path.basename(video_path)}] {frame_idx}/{total_frames} frames")

    if batch:
        all_frames.extend(extract_landmarks_batch(batch, wholebody))

    cap.release()

    arr = np.array(all_frames, dtype=np.float32)  # (T, 59, 3)

    arr = interpolate_missing_hands(arr)

    left_missing  = np.all(arr[:, 17:38] == 0, axis=(1, 2))
    right_missing = np.all(arr[:, 38:59] == 0, axis=(1, 2))
    mask = ~(left_missing | right_missing)
    arr  = arr[mask]

    kept  = int(mask.sum())
    total = len(mask)
    pct   = 100 * kept / total if total > 0 else 0
    print(f"  Kept {kept}/{total} frames ({pct:.1f}%) after filtering")

    if kept < 30:
        print(f"  SKIPPING — too few usable frames ({kept})")
        return pct

    class_name = os.path.basename(os.path.dirname(video_path))
    save_dir   = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(save_dir, exist_ok=True)
    save_path  = os.path.join(save_dir,
                              f"{os.path.splitext(os.path.basename(video_path))[0]}.npy")
    np.save(save_path, arr)
    print(f"  Saved {save_path} | Shape: {arr.shape}")

    return pct

In [8]:
def main(path):
    report_df    = pd.read_csv("VideoReport.csv")
    valid_videos = set(
        report_df[
            report_df["duration_ok"] &
            report_df["resolution_ok"] &
            #report_df["blur_ok"] & #Ignore blur (manually checked)
            report_df["brightness_ok"] &
            report_df["fps_ok"]
        ]["video"].astype(str)
    )

    #filter out vids
    for v in report_df[~report_df["video"].astype(str).isin(valid_videos)]["video"].tolist():
        print(f"Skipping (filtered out): {v}")
    

    wholebody = Wholebody(
        mode="performance",
        backend="onnxruntime",
        device="cuda",
        to_openpose=False,
    )
    for root, _, files in os.walk(path):
        for fname in sorted(files):
            if not fname.lower().endswith(".mp4"):
                continue
            rel  = os.path.relpath(os.path.join(root, fname), path)
            full = os.path.join(root, fname)
            if rel not in valid_videos:
                print(f"Skipping: {rel}")
                continue
            pct = extract_landmarks_from_video(full, wholebody)
            if pct is not None:
                idx = report_df[report_df["video"].astype(str) == rel].index[0]
                report_df.loc[idx, "two_hand_pct"] = round(pct, 2)
                report_df.loc[idx, "two_hand_ok"] = pct >= 50
            
    report_df.to_csv("VideoReport_updated.csv", index=False)
    print("Processing complete. Updated report saved as VideoReport_updated.csv")

In [9]:
main(PATH)

Skipping (filtered out): 1-Ardhpatka\VID_20250529_153542.mp4
Skipping (filtered out): 19-Chatura\IMG_1246.mp4
Skipping (filtered out): 5-Aarala\IMG_1161.mp4
load C:\Users\Vivek\.cache\rtmlib\hub\checkpoints\yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend
load C:\Users\Vivek\.cache\rtmlib\hub\checkpoints\rtmw-dw-x-l_simcc-cocktail14_270e-384x288_20231122.onnx with onnxruntime backend
Processing: data\Videos\1-Ardhpatka\IMG_1134.mp4
  [IMG_1134.mp4] 100/346 frames
  [IMG_1134.mp4] 200/346 frames
  [IMG_1134.mp4] 300/346 frames
  Kept 346/346 frames (100.0%) after filtering
  Saved landmarks\1-Ardhpatka\IMG_1134.npy | Shape: (346, 59, 3)
Processing: data\Videos\1-Ardhpatka\IMG_1135.mp4
  [IMG_1135.mp4] 100/340 frames
  [IMG_1135.mp4] 200/340 frames
  [IMG_1135.mp4] 300/340 frames
  Kept 340/340 frames (100.0%) after filtering
  Saved landmarks\1-Ardhpatka\IMG_1135.npy | Shape: (340, 59, 3)
Processing: data\Videos\1-Ardhpatka\IMG_1136.mp4
  [IMG_1136.mp4] 100/343 frames
 